# SuperKart sales estimation

Estimate product–store sales totals using tabular regression. The data has no sales period, so weekly, monthly, quarterly, and annual forecasts require dated sales history.

## 1. Set up the project

This notebook clones the public GitHub repository, installs its dependencies, and runs each project stage in order.


In [ ]:
!git clone https://github.com/sravanml/superkart-sales-mlops.git /content/superkart
%cd /content/superkart
%pip install -q -r requirements.txt


## 2. Data registration

The validator checks the expected schema and reports completeness, duplicates, and target statistics.

In [ ]:
import subprocess, sys
def stage(name):
    subprocess.run([sys.executable, 'pipeline.py', name], check=True)
stage('validate')

## 3. Data preparation

The source has 8,763 unique products across four stores. Standardize the sugar label `reg` to `Regular` and omit the unique product ID. Use an 80/20 split stratified by store to evaluate new products at known stores.

In [ ]:
stage('prepare')

## 4. Training and experiment tracking

Tune Random Forest depth indirectly via minimum leaf size and vary the number of trees. Use three-fold cross-validation with MAE and compare the winner against a median baseline on the holdout.

In [ ]:
stage('train')

## 5. Results and inference

 Compare test MAE, RMSE, and R² with the baseline. 

In [ ]:
import joblib, json, pandas as pd
from pipeline import FEATURES, MODEL
results = json.loads((MODEL.parent / 'metrics.json').read_text())
display(pd.DataFrame([results['baseline'], results['random_forest']], index=['Median baseline', 'Random Forest']))
delta = results['baseline']['MAE'] - results['random_forest']['MAE']
print(f'Test MAE improvement over baseline: {delta:,.2f} sales units')
artifact = joblib.load(MODEL)
example = pd.read_csv('splits/test.csv').head(1)
example['Estimated_Sales'] = artifact['pipeline'].predict(example[FEATURES])
display(example)

## 6. Automation and deployment

GitHub Actions validates the source, saves the train/test files as a workflow artifact, trains from those files, then commits the resulting model and evaluation files to main. The Streamlit app loads the committed model. Create a public Hugging Face Space using Docker, push Dockerfile, source files, and the trained models/ folder, then verify that the app opens publicly.

- Public GitHub repository: ADD URL
- Successful GitHub Actions run: ADD URL
- Public Hugging Face Space: ADD URL
- Streamlit Community Cloud (if deployed): ADD URL

Add screenshots showing the repository folder structure, successful workflow run, and public deployed app. Run all notebook code cells from top to bottom and keep their outputs visible before submission.

## 7. Conclusion

The holdout estimates performance for new products within the four observed stores. Evaluating new stores would require a store-based holdout. Forecasts by calendar period require dated sales observations.